<a href="https://colab.research.google.com/github/Vanshitamehta06/Genai/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# =============================
# INSTALL
# =============================
!pip uninstall -y transformers peft accelerate trl
!pip install -q transformers==4.40.2 peft==0.9.0 accelerate==0.29.3 trl==0.8.6 datasets torch evaluate

# =============================
# IMPORTS
# =============================
from google.colab import drive
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer
import torch
import shutil

# =============================
# MOUNT DRIVE
# =============================
drive.mount('/content/drive')

# =============================
# LOAD + SPLIT DATASET
# =============================
dataset = load_dataset(
    "json",
    data_files="/content/train_dataset.json"
)["train"]

dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

# =============================
# LOAD MODEL
# =============================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="auto"
)

# =============================
# LORA
# =============================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# =============================
# TRAINING ARGS
# =============================
training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="epoch",
    fp16=False,
    report_to="none"
)

# =============================
# TRAINER
# =============================
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field="text",
    packing=True
)

# =============================
# TRAIN
# =============================
print("🚀 Training started...")
trainer.train()

# =============================
# SAVE MODEL
# =============================
save_path = "/content/final_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

# SAVE TO DRIVE
shutil.copytree(save_path, "/content/drive/MyDrive/final_model", dirs_exist_ok=True)

print("✅ Model saved")

Found existing installation: transformers 4.40.2
Uninstalling transformers-4.40.2:
  Successfully uninstalled transformers-4.40.2
Found existing installation: peft 0.9.0
Uninstalling peft-0.9.0:
  Successfully uninstalled peft-0.9.0
Found existing installation: accelerate 0.29.3
Uninstalling accelerate-0.29.3:
  Successfully uninstalled accelerate-0.29.3
Found existing installation: trl 0.8.6
Uninstalling trl-0.8.6:
  Successfully uninstalled trl-0.8.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.2 which is incompatible.


ImportError: cannot import name 'is_tf_available' from 'transformers.utils' (/usr/local/lib/python3.12/dist-packages/transformers/utils/__init__.py)

In [2]:
# =============================
# INSTALL (if needed)
# =============================
!pip install -q evaluate

# =============================
# IMPORTS
# =============================
import evaluate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset

# =============================
# LOAD DATASET (TEST SPLIT)
# =============================
dataset = load_dataset(
    "json",
    data_files="/content/train_dataset.json"
)["train"]

dataset = dataset.train_test_split(test_size=0.1)
test_dataset = dataset["test"]

print("✅ Test size:", len(test_dataset))

# =============================
# LOAD METRICS
# =============================
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# =============================
# PATHS
# =============================
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ft_model_path = "/content/final_model"

# =============================
# TOKENIZER
# =============================
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# =============================
# LOAD BASE MODEL (SEPARATE)
# =============================
import os
os.makedirs("/content/offload", exist_ok=True)

# BASE MODEL
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    offload_folder="/content/offload"
)

# FINETUNED BASE
ft_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    offload_folder="/content/offload"
)

# APPLY LORA (FIXED)
ft_model = PeftModel.from_pretrained(
    ft_base_model,
    ft_model_path,
    device_map="auto",
    offload_folder="/content/offload"
)

print("✅ Models loaded without error")

print("✅ Both models loaded correctly")

# =============================
# GENERATE FUNCTION
# =============================
def generate(model, prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =============================
# SPLIT FUNCTION (IMPORTANT)
# =============================
def split_text(text):
    if "Answer:" in text:
        parts = text.split("Answer:")
        prompt = parts[0] + "Answer:"
        answer = parts[1].strip()
    else:
        prompt = text
        answer = text

    return prompt, answer

# =============================
# EVALUATION LOOP
# =============================
test_samples = test_dataset.select(range(30))

references = []
base_preds = []
ft_preds = []

for sample in test_samples:
    text = sample["text"]

    prompt, reference = split_text(text)

    base_out = generate(base_model, prompt)
    ft_out = generate(ft_model, prompt)

    references.append(reference)
    base_preds.append(base_out)
    ft_preds.append(ft_out)

# =============================
# METRICS
# =============================
base_bleu = bleu.compute(
    predictions=base_preds,
    references=[[r] for r in references]
)

ft_bleu = bleu.compute(
    predictions=ft_preds,
    references=[[r] for r in references]
)

base_rouge = rouge.compute(
    predictions=base_preds,
    references=references
)

ft_rouge = rouge.compute(
    predictions=ft_preds,
    references=references
)

# =============================
# RESULTS
# =============================
print("\n📊 BASE MODEL BLEU:", base_bleu)
print("📊 FINETUNED MODEL BLEU:", ft_bleu)

print("\n📊 BASE MODEL ROUGE:", base_rouge)
print("📊 FINETUNED MODEL ROUGE:", ft_rouge)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00


SystemError: rw-p 000() method: bad call flags

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os
os.listdir("/content/drive/MyDrive")

['phone memory',
 'Memories',
 'IMG-20250303-WA0155.jpg',
 'IMG-20250303-WA0153.jpg',
 'IMG-20250303-WA0151.jpg',
 'IMG-20250303-WA0149.jpg',
 'IMG-20250303-WA0147.jpg',
 'IMG-20250303-WA0145.jpg',
 'IMG-20250303-WA0143.jpg',
 'IMG-20250303-WA0141.jpg',
 'VID-20250303-WA0140.mp4',
 'IMG-20250303-WA0138.jpg',
 'IMG-20250303-WA0136.jpg',
 'IMG-20250303-WA0134.jpg',
 'IMG-20250303-WA0132.jpg',
 'IMG-20250303-WA0130.jpg',
 'IMG-20250303-WA0128.jpg',
 'IMG-20250303-WA0127.jpg',
 'IMG-20250303-WA0126.jpg',
 'IMG-20250303-WA0123.jpg',
 'IMG-20250303-WA0122.jpg',
 'IMG-20250303-WA0119.jpg',
 'IMG-20250303-WA0117.jpg',
 'IMG-20250303-WA0115.jpg',
 'IMG-20250303-WA0114.jpg',
 'IMG-20250303-WA0111.jpg',
 '20250303_131621.jpg',
 'VID-20250303-WA0109.mp4',
 'IMG-20250303-WA0108.jpg',
 'IMG-20250303-WA0107.jpg',
 'VID-20250303-WA0106.mp4',
 'VID-20250303-WA0105.mp4',
 'VID-20250303-WA0104.mp4',
 'VID-20250303-WA0103.mp4',
 'VID-20250303-WA0102.mp4',
 'VID-20250303-WA0101.mp4',
 'VID-20250303-WA0100.

In [12]:
pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=e3a928976fa435a8b64a2b0256995f5031271a0a7faf8c8cc26935883e92083a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [13]:
# =============================
# INSTALL
# =============================
!pip install -q evaluate

# =============================
# IMPORTS
# =============================
import os
import torch
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from google.colab import drive

# =============================
# MOUNT DRIVE
# =============================
drive.mount('/content/drive')

# =============================
# DEBUG: CHECK FILES (IMPORTANT)
# =============================
print("📂 Drive files:")
print(os.listdir("/content/drive/MyDrive"))

# =============================
# FIX DATASET PATH (EDIT IF NEEDED)
# =============================
DATASET_PATH = "/content/train_dataset.json"

# =============================
# LOAD DATASET
# =============================
dataset = load_dataset(
    "json",
    data_files=DATASET_PATH
)["train"]

dataset = dataset.train_test_split(test_size=0.1)
test_dataset = dataset["test"]

print("✅ Test size:", len(test_dataset))

# =============================
# LOAD METRICS
# =============================
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# =============================
# MODEL PATHS (FROM DRIVE)
# =============================
base_model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ft_model_path = "/content/drive/MyDrive/final_model"   # 🔥 YOUR SAVED MODEL

# =============================
# CREATE OFFLOAD FOLDER
# =============================
os.makedirs("/content/offload", exist_ok=True)

# =============================
# TOKENIZER
# =============================
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# =============================
# LOAD BASE MODEL
# =============================
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    offload_folder="/content/offload"
)

# =============================
# LOAD FINETUNED MODEL (FROM DRIVE)
# =============================
ft_base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    device_map="auto",
    offload_folder="/content/offload"
)

ft_model = PeftModel.from_pretrained(
    ft_base_model,
    ft_model_path,
    device_map="auto",
    offload_folder="/content/offload"
)

print("✅ Models loaded from Drive")

# =============================
# GENERATE FUNCTION
# =============================
def generate(model, prompt):
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# =============================
# SPLIT FUNCTION
# =============================
def split_text(text):
    if "Answer:" in text:
        parts = text.split("Answer:")
        prompt = parts[0] + "Answer:"
        answer = parts[1].strip()
    else:
        prompt = text
        answer = text
    return prompt, answer

# =============================
# EVALUATION LOOP
# =============================
test_samples = test_dataset.select(range(10))

references = []
base_preds = []
ft_preds = []

for i, sample in enumerate(test_samples):
    text = sample["text"]

    prompt, reference = split_text(text)

    base_out = generate(base_model, prompt)
    ft_out = generate(ft_model, prompt)

    references.append(reference)
    base_preds.append(base_out)
    ft_preds.append(ft_out)

    print("\n" + "="*80)
    print(f"🔹 SAMPLE {i+1}")

    print("\n🧾 QUESTION:")
    print(prompt)

    print("\n🤖 BASE MODEL OUTPUT:")
    print(base_out)

    print("\n🚀 FINETUNED MODEL OUTPUT:")
    print(ft_out)

    print("\n✅ EXPECTED ANSWER:")
    print(reference)

# =============================
# METRICS
# =============================
base_bleu = bleu.compute(
    predictions=base_preds,
    references=[[r] for r in references]
)

ft_bleu = bleu.compute(
    predictions=ft_preds,
    references=[[r] for r in references]
)

base_rouge = rouge.compute(
    predictions=base_preds,
    references=references
)

ft_rouge = rouge.compute(
    predictions=ft_preds,
    references=references
)

# =============================
# FINAL RESULTS
# =============================
print("\n" + "="*80)
print("📊 FINAL RESULTS")

print("\n📊 BASE MODEL BLEU:", base_bleu)
print("📊 FINETUNED MODEL BLEU:", ft_bleu)

print("\n📊 BASE MODEL ROUGE:", base_rouge)
print("📊 FINETUNED MODEL ROUGE:", ft_rouge)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 Drive files:
['phone memory', 'Memories', 'IMG-20250303-WA0155.jpg', 'IMG-20250303-WA0153.jpg', 'IMG-20250303-WA0151.jpg', 'IMG-20250303-WA0149.jpg', 'IMG-20250303-WA0147.jpg', 'IMG-20250303-WA0145.jpg', 'IMG-20250303-WA0143.jpg', 'IMG-20250303-WA0141.jpg', 'VID-20250303-WA0140.mp4', 'IMG-20250303-WA0138.jpg', 'IMG-20250303-WA0136.jpg', 'IMG-20250303-WA0134.jpg', 'IMG-20250303-WA0132.jpg', 'IMG-20250303-WA0130.jpg', 'IMG-20250303-WA0128.jpg', 'IMG-20250303-WA0127.jpg', 'IMG-20250303-WA0126.jpg', 'IMG-20250303-WA0123.jpg', 'IMG-20250303-WA0122.jpg', 'IMG-20250303-WA0119.jpg', 'IMG-20250303-WA0117.jpg', 'IMG-20250303-WA0115.jpg', 'IMG-20250303-WA0114.jpg', 'IMG-20250303-WA0111.jpg', '20250303_131621.jpg', 'VID-20250303-WA0109.mp4', 'IMG-20250303-WA0108.jpg', 'IMG-20250303-WA0107.jpg', 'VID-20250303-WA0106.mp4', 'VID-20250303-WA0105.mp4', 'VID-20250303-WA0104.

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/final_model'